# Reproduce the order-book benchmark

Use this project's virtual environment as the notebook kernel. Jupyter is an optional viewer, not needed by the CLI. All training runs on CPU. This notebook executes the same public commands as the README and reads their raw artifacts.


In [ ]:
from pathlib import Path
import json, subprocess, sys
root = Path.cwd()
if root.name == "benchmarks":
    root = root.parent
assert (root / "configs/demo.yaml").exists(), "Open this notebook inside project 07"


In [ ]:
subprocess.run([sys.executable, "benchmarks/reproduce.py", "--config", "configs/demo.yaml", "--output", "reports/notebook.html"], cwd=root, check=True)


## Accuracy, costs and measured overhead

The report groups only matching data/source/configuration/runtime protocols. Confidence intervals describe seed variability on one synthetic dataset. They do not establish performance on real markets.


In [ ]:
from IPython.display import HTML, display
display(HTML((root / "reports/notebook.html").read_text()))


In [ ]:
results = json.loads((root / "reports/notebook.json").read_text())
assert not results["failures"]
for result in results["summary"]:
    print(result["model"], result["qubits"], result["encoding"], "balanced accuracy:", round(result["balanced_accuracy"], 3), "net bps/fold:", round(result["net_bps"], 2))


## Audit the actual chronological windows

Raw predictions, trades, latency samples and model artifacts are siblings of each run manifest. Test data never chooses preprocessing, temperature or decision threshold.


In [ ]:
manifests = [json.loads(Path(p).read_text()) for p in results["runs"]]
for manifest in manifests:
    for split in manifest["splits"]:
        for left, right in zip(["train", "calibration", "validation"], ["calibration", "validation", "test"]):
            assert max(split[left]) + manifest["config"]["horizon"] < min(split[right]) - manifest["config"]["embargo"]
    print(manifest["solver"], manifest["status"], len(manifest["results"]), "trials", manifest["wall_seconds"], "seconds")


## Shot ablation

The finite-shot kernel uses binomial estimates of analytic overlaps. VQC shots use PennyLane measurements. Both are local simulation, with no hardware queue, communication or physical noise.


In [ ]:
subprocess.run([sys.executable, "benchmarks/reproduce.py", "--config", "configs/shots.yaml", "--output", "reports/notebook-shots.html"], cwd=root, check=True)
display(HTML((root / "reports/notebook-shots.html").read_text()))
